# S07 · Fit a line, then watch it learn

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GirishMKulkarni/applied-maths-in-industry-site/blob/main/sessions/S07/notebooks/01_fit_a_line.ipynb)

In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
# Load the Boston Housing dataset
boston = fetch_openml(name='boston', version=1, as_frame=True)
X = boston.data  # Features
y = boston.target  # Target variable (house prices)

In [ ]:
# Select a single feature for simplicity: 'RM' (average number of rooms)
X_rm = X['RM'].values.reshape(-1, 1)

In [ ]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_rm, y, test_size=0.3, random_state=42)

In [ ]:
# Transform the feature to include polynomial terms (degree=2)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

In [ ]:
# Fit a linear regression model to the polynomial features
model = LinearRegression()
model.fit(X_train_poly, y_train)

In [ ]:
# Make predictions on the test set
y_pred = model.predict(X_test_poly)

In [ ]:
# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f'Mean Squared Error: {mse:.2f}')
print(f'R² Score: {r2:.2f}')

In [ ]:
# Plot the results
plt.scatter(X_test, y_test, color='blue', label='Actual Prices')
plt.scatter(X_test, y_pred, color='red', label='Predicted Prices')
plt.xlabel('Average Number of Rooms (RM)')
plt.ylabel('House Price')
plt.title('Polynomial Regression (Degree=2) on Boston Housing Dataset')
plt.legend()
plt.show()

## Learning rate — how the model actually finds that line

`model.fit()` solved for the line in one shot. Underneath, most learning algorithms find it by taking small steps downhill instead — nudging the slope and intercept a little on every step. How big each step is is the **learning rate**. Too small and it crawls; too big and it overshoots and blows up; just right and it settles quickly on the same answer.

In [ ]:
# Same single feature (RM), scaled so the steps behave.
x = (X_train.flatten() - X_train.mean()) / X_train.std()
yt = y_train.values

def predict(w, b):
    return w * x + b

def mse_of(w, b):
    return ((yt - predict(w, b)) ** 2).mean()

def gradient(w, b):
    error = predict(w, b) - yt
    return 2 * np.mean(error * x), 2 * np.mean(error)

In [ ]:
# Try a few learning rates, 100 steps each, starting from w=0, b=0.
for lr in [0.001, 0.01, 0.3, 1.2]:
    w, b = 0.0, 0.0
    for _ in range(100):
        gw, gb = gradient(w, b)
        w -= lr * gw
        b -= lr * gb
    print(f"learning rate {lr:<6} -> MSE after 100 steps: {mse_of(w, b):,.2f}")

In [ ]:
# Learning rate 0.3 settles nicely -- watch the score fall step by step.
w, b, lr = 0.0, 0.0, 0.3
history = []
for _ in range(300):
    gw, gb = gradient(w, b)
    w -= lr * gw
    b -= lr * gb
    history.append(mse_of(w, b))

plt.plot(history)
plt.xlabel('step')
plt.ylabel('MSE')
plt.title(f'Learning rate = {lr}')
plt.show()

print('gradient descent slope/intercept (scaled RM):', round(w, 3), round(b, 3))

In [ ]:
# Check: does gradient descent land where LinearRegression lands?
check = LinearRegression().fit(x.reshape(-1, 1), yt)
print('sklearn        :', round(check.coef_[0], 3), round(check.intercept_, 3))
print('gradient descent:', round(w, 3), round(b, 3))